ETL Code

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================================
# 1. EXTRACT (Data Ingestion)
# ==========================================================
def load_raw_data():
    """Extracts raw data from disparate sources."""
    
    base = Path(r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download")
    
    sales = pd.read_csv(base / "Anonymized_Restaurant_Sales_Data.csv", encoding="latin1")
    pnl = pd.read_csv(base / "Public_Derived_Consolidated_PnL_2023_2025.csv", encoding="latin1")
    digital = pd.read_csv(base / "Public_DigitalChannels_Monthly_Revenue_FinancialYear.csv", encoding="latin1")
    epos = pd.read_csv(base / "Public_EPOS_Category_Summary_2023_2025.csv", encoding="latin1")

    return sales, pnl, digital, epos


# ==========================================================
# 2. TRANSFORM (Feature Engineering & Cleaning)
# ==========================================================
def transform_data(sales_df, pnl_df):
    """Performs cleaning and temporal feature engineering."""
    
    # --- Fix P&L column name ---
    value_col = [c for c in pnl_df.columns if "Value" in c][0]

    # --- Temporal Transformation ---
    sales_df['Date'] = pd.to_datetime(sales_df['Date'], dayfirst=True)
    sales_df['Hour'] = pd.to_datetime(sales_df['Time'], format='%H:%M').dt.hour
    sales_df['Is_Weekend'] = sales_df['Date'].dt.dayofweek.isin([5, 6]).astype(int)

    # --- Data Cleaning ---
    sales_clean = sales_df[sales_df['Cancelled'] == 'No'].copy()

    # --- P&L Normalisation ---
    pnl_clean = pnl_df.dropna(subset=['Account Category', value_col]).copy()
    pnl_clean['Revenue %'] = pnl_clean['Revenue %'].str.replace('%', '').astype(float) / 100

    return sales_clean, pnl_clean, value_col


# ==========================================================
# 3. LOAD (The 5 Strategic Insight Layers)
# ==========================================================
def generate_insight_layers(sales_clean, pnl_clean, value_col, digital, epos_summary):
    """Generates the 5 structured CSVs for the Agentic AI."""
    
    # 1. Category Performance
    cat_perf = sales_clean.groupby('Category').agg({
        'Gross Sales': 'sum', 'Quantity': 'sum', 'Est. Profit': 'sum'
    }).reset_index()
    cat_perf['Profit_Margin'] = cat_perf['Est. Profit'] / cat_perf['Gross Sales']
    cat_perf.to_csv('Processed_Category_Performance.csv', index=False)

    # 2. Hourly Demand Patterns
    hourly = sales_clean.groupby(['DayOfWeek', 'Hour']).agg({'Gross Sales': 'sum'}).reset_index()
    hourly.to_csv('Processed_Hourly_Sales.csv', index=False)

    # 3. Item Profitability
    item_perf = sales_clean.groupby(['Category', 'Line item name']).agg({
        'Gross Sales': 'sum', 'Quantity': 'sum', 'Est. Profit': 'sum', 'Price Per Item': 'mean'
    }).reset_index()
    item_perf['Margin_Percentage'] = (item_perf['Est. Profit'] / item_perf['Gross Sales']) * 100
    item_perf.to_csv('Processed_Item_Performance.csv', index=False)

    # 4. Scenario Modelling
    rev = pnl_clean[pnl_clean['Account Category'] == 'Operating Revenue'][value_col].sum()
    v_cost = abs(pnl_clean[pnl_clean['Account Category'] == 'Total Variable Costs (COGS)'][value_col].sum())
    f_cost = abs(pnl_clean[pnl_clean['Account Category'] == 'Operating Expenses (Fixed)'][value_col].sum())

    scenario = pd.DataFrame([{
        'Total_Revenue': rev,
        'Variable_Cost_Ratio': v_cost / rev,
        'Fixed_Costs': f_cost,
        'Break_Even_Point': f_cost / (1 - (v_cost / rev))
    }])
    scenario.to_csv('Scenario_Modelling_Base.csv', index=False)

    # 5. Multi-Channel Revenue Intelligence
    digital_sum = digital.groupby('Channel').agg({'Revenue': 'sum'}).reset_index()
    epos_total = epos_summary['Gross_Sales'].sum()
    epos_row = pd.DataFrame([{'Channel': 'EPOS (In-Store)', 'Revenue': epos_total}])

    multi_channel = pd.concat([digital_sum, epos_row], ignore_index=True)
    multi_channel['Contribution_Percentage'] = (multi_channel['Revenue'] / multi_channel['Revenue'].sum()) * 100
    multi_channel.to_csv('Multi_Channel_Revenue_Insight.csv', index=False)

    return "ETL COMPLETE: All 5 Insight Layers Generated Successfully"


# ==========================================================
# 4. EXECUTION
# ==========================================================
raw_s, raw_p, raw_d, raw_e = load_raw_data()
clean_s, clean_p, value_col = transform_data(raw_s, raw_p)
status = generate_insight_layers(clean_s, clean_p, value_col, raw_d, raw_e)

print(status)



ETL COMPLETE: All 5 Insight Layers Generated Successfully


Metrics Extractor

In [22]:
import pandas as pd
def load_insight_layers():
    """Loads the 5 processed CSVs generated by the ETL pipeline."""
    
    cat = pd.read_csv(
        r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Agentic_AI_Project\Processed_Category_Performance.csv"
    )
    hourly = pd.read_csv(
        r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Agentic_AI_Project\Processed_Hourly_Sales.csv"
    )
    item = pd.read_csv(
        r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Agentic_AI_Project\Processed_Item_Performance.csv"
    )
    scenario = pd.read_csv(
        r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Agentic_AI_Project\Scenario_Modelling_Base.csv"
    )
    channels = pd.read_csv(
        r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Agentic_AI_Project\Multi_Channel_Revenue_Insight.csv"
    )
    
    return cat, hourly, item, scenario, channels
, channels


def extract_metrics():
    """Extracts high-value business metrics for the Agentic AI reasoning engine."""
    
    cat, hourly, item, scenario, channels = load_insight_layers()
    
    metrics = {}

    # ==========================================================
    # 1. CATEGORY PERFORMANCE
    # ==========================================================
    top_cat = cat.loc[cat['Gross Sales'].idxmax()]
    worst_cat = cat.loc[cat['Gross Sales'].idxmin()]
    
    metrics["category"] = {
        "top_category": top_cat["Category"],
        "top_category_revenue": float(top_cat["Gross Sales"]),
        "worst_category": worst_cat["Category"],
        "worst_category_revenue": float(worst_cat["Gross Sales"]),
        "average_margin": float(cat["Profit_Margin"].mean())
    }

    # ==========================================================
    # 2. HOURLY SALES (OPERATIONAL INTELLIGENCE)
    # ==========================================================
    peak_hour = hourly.loc[hourly['Gross Sales'].idxmax()]
    slow_hour = hourly.loc[hourly['Gross Sales'].idxmin()]
    
    metrics["operations"] = {
        "peak_hour": int(peak_hour["Hour"]),
        "peak_hour_sales": float(peak_hour["Gross Sales"]),
        "slowest_hour": int(slow_hour["Hour"]),
        "slowest_hour_sales": float(slow_hour["Gross Sales"])
    }

    # ==========================================================
    # 3. ITEM PERFORMANCE (FINANCIAL INTELLIGENCE)
    # ==========================================================
    top_item = item.loc[item['Margin_Percentage'].idxmax()]
    low_item = item.loc[item['Margin_Percentage'].idxmin()]
    
    metrics["items"] = {
        "highest_margin_item": top_item["Line item name"],
        "highest_margin_value": float(top_item["Margin_Percentage"]),
        "lowest_margin_item": low_item["Line item name"],
        "lowest_margin_value": float(low_item["Margin_Percentage"])
    }

    # ==========================================================
    # 4. SCENARIO MODELLING (STRATEGIC INTELLIGENCE)
    # ==========================================================
    metrics["scenario"] = {
        "total_revenue": float(scenario["Total_Revenue"].iloc[0]),
        "variable_cost_ratio": float(scenario["Variable_Cost_Ratio"].iloc[0]),
        "fixed_costs": float(scenario["Fixed_Costs"].iloc[0]),
        "break_even_point": float(scenario["Break_Even_Point"].iloc[0])
    }

    # ==========================================================
    # 5. MULTI-CHANNEL REVENUE (MARKET INTELLIGENCE)
    # ==========================================================
    top_channel = channels.loc[channels['Revenue'].idxmax()]
    low_channel = channels.loc[channels['Revenue'].idxmin()]
    
    metrics["channels"] = {
        "top_channel": top_channel["Channel"],
        "top_channel_revenue": float(top_channel["Revenue"]),
        "lowest_channel": low_channel["Channel"],
        "lowest_channel_revenue": float(low_channel["Revenue"]),
        "channel_mix": channels.set_index("Channel")["Contribution_Percentage"].to_dict()
    }

    return metrics


Insight Generator

In [8]:
# Step 1 - Build Rule-Based Insight Functions to analyse the metrics before LLM sees them

In [42]:
def rule_based_insights(metrics):
    insights = []

    # CATEGORY INSIGHTS
    cat = metrics["category"]
    if cat["top_category_revenue"] > 2 * cat["worst_category_revenue"]:
        insights.append(f"Category '{cat['top_category']}' significantly outperforms '{cat['worst_category']}'.")
    if cat["average_margin"] < 0.4:
        insights.append("Average category margin is low — consider pricing adjustments.")

    # OPERATIONS INSIGHTS
    ops = metrics["operations"]
    if ops["peak_hour"] < 15:
        insights.append("Peak demand occurs earlier than typical restaurant patterns.")
    if ops["slowest_hour"] > 19:
        insights.append("Evening sales are unusually weak compared to industry norms.")

    # ITEM INSIGHTS
    items = metrics["items"]
    if items["highest_margin_value"] - items["lowest_margin_value"] < 5:
        insights.append("Item margins are very consistent — limited pricing differentiation.")
    else:
        insights.append(f"'{items['lowest_margin_item']}' may require repricing due to low margin.")

    # CHANNEL INSIGHTS
    ch = metrics["channels"]
    if ch["top_channel_revenue"] > 2 * ch["lowest_channel_revenue"]:
        insights.append(f"Strong dependency on '{ch['top_channel']}' — consider diversifying channels.")

    # SCENARIO INSIGHTS
    sc = metrics["scenario"]
    if sc["break_even_point"] > sc["total_revenue"] * 0.7:
        insights.append("Break-even point is high relative to revenue — fixed costs may be too high.")

    return insights


In [44]:
#Step 2 - Build Anomaly Detection
def detect_anomalies(metrics):
    anomalies = []

    # Operational anomaly
    ops = metrics["operations"]
    if ops["peak_hour"] > 20 or ops["peak_hour"] < 10:
        anomalies.append("Peak hour is outside typical restaurant operating patterns.")

    # Channel anomaly
    ch = metrics["channels"]
    if ch["channel_mix"][ch["top_channel"]] > 50:
        anomalies.append("More than half of revenue comes from a single channel.")

    # Margin anomaly
    cat = metrics["category"]
    if cat["average_margin"] > 0.75:
        anomalies.append("Margins appear unusually high — check cost allocation accuracy.")

    return anomalies


Updated Insight Generator using Simlulated LLM

In [32]:
#Step 1 Build Simulated LLM with full narrative analysis
def llm(prompt, metrics=None):
    """
    Simulated LLM that generates realistic insights without an API.
    Uses the metrics dictionary to produce narrative analysis.
    """

    cat = metrics["category"]
    ops = metrics["operations"]
    items = metrics["items"]
    sc = metrics["scenario"]
    ch = metrics["channels"]

    return f"""
SUMMARY:
The business shows strong performance in the '{cat['top_category']}' category, generating £{cat['top_category_revenue']:.2f}, 
while '{cat['worst_category']}' is significantly underperforming. Operational demand peaks at {ops['peak_hour']}:00, 
with the slowest period at {ops['slowest_hour']}:00. Item-level profitability is consistent, with margins ranging 
from {items['lowest_margin_value']:.1f}% to {items['highest_margin_value']:.1f}%. The break-even point is £{sc['break_even_point']:.2f}, 
driven by fixed costs of £{sc['fixed_costs']:.2f}. EPOS (In-Store) remains the dominant revenue channel.

KEY INSIGHTS:
- '{cat['top_category']}' is the strongest category, outperforming '{cat['worst_category']}' by a wide margin.
- Peak demand at {ops['peak_hour']}:00 suggests earlier customer behaviour than typical restaurants.
- '{items['highest_margin_item']}' delivers the highest margin, while '{items['lowest_margin_item']}' is the weakest.
- Break-even point represents {sc['break_even_point'] / sc['total_revenue'] * 100:.1f}% of total revenue.
- EPOS contributes {ch['channel_mix']['EPOS (In-Store)']:.1f}% of total revenue, indicating strong in-store performance.

ANOMALIES:
- Evening sales are unusually weak, with the slowest hour at {ops['slowest_hour']}:00.
- Revenue dependency on EPOS is high at {ch['channel_mix']['EPOS (In-Store)']:.1f}%.
- Category margins are unusually consistent, suggesting limited pricing differentiation.

RECOMMENDATIONS:
1. Promote high-margin items such as '{items['highest_margin_item']}' to increase contribution margin.
2. Review pricing or cost structure for '{items['lowest_margin_item']}' to improve profitability.
3. Strengthen weaker channels like '{ch['lowest_channel']}' to reduce dependency on EPOS.
4. Introduce targeted evening promotions to address weak performance after {ops['slowest_hour']}:00.
5. Reassess fixed costs to reduce the break-even threshold.

RISKS & OPPORTUNITIES:
- Risk: Over-reliance on EPOS may expose the business to operational disruptions.
- Opportunity: Website and Deliveroo channels show potential for growth with targeted marketing.
- Opportunity: Menu optimisation could further improve already strong margins.
"""

# Step 2 Pass metrics to the simlulated LLM
insights = generate_insights(metrics, lambda prompt: llm(prompt, metrics))

# Run the full pipeline
metrics = extract_metrics()
print(metrics)
insights = generate_insights(metrics, lambda prompt: llm(prompt, metrics))
print(insights["llm_output"])


{'category': {'top_category': 'EXTRAS', 'top_category_revenue': 12777.5, 'worst_category': 'Friday Specials', 'worst_category_revenue': 125.5, 'average_margin': 0.6500736134823106}, 'operations': {'peak_hour': 16, 'peak_hour_sales': 3478.0, 'slowest_hour': 20, 'slowest_hour_sales': 542.0}, 'items': {'highest_margin_item': 'Chin Chin', 'highest_margin_value': 65.14193548387097, 'lowest_margin_item': 'Maltina', 'lowest_margin_value': 64.91286307053942}, 'scenario': {'total_revenue': 287424.62, 'variable_cost_ratio': 0.234974651788702, 'fixed_costs': 164400.4, 'break_even_point': 214895.3631201682}, 'channels': {'top_channel': 'EPOS (In-Store)', 'top_channel_revenue': 75636.84000000001, 'lowest_channel': 'Deliveroo', 'lowest_channel_revenue': 29290.37, 'channel_mix': {'Deliveroo': 14.152776641613167, 'JustEat': 30.611218762875463, 'Website': 18.68913603777608, 'EPOS (In-Store)': 36.54686855773528}}}

SUMMARY:
The business shows strong performance in the 'EXTRAS' category, generating £1277

# Next step below is the forecasting layer - development and implementation

In [34]:
import pandas as pd

# ---------------------------------------------------------
# Helper: safely parse timestamps ONLY if present
# ---------------------------------------------------------
def try_parse_timestamp(df, possible_cols=["timestamp", "datetime", "date", "time"]):
    for col in possible_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            df = df.dropna(subset=[col])
            df = df.sort_values(col)
            df = df.rename(columns={col: "timestamp"})
            return df
    # If no timestamp column exists, return unchanged
    return df


# ---------------------------------------------------------
# Load all 5 CSVs safely
# ---------------------------------------------------------
def load_all_data():
    data = {}

    # 1. Multi-channel revenue (no timestamp expected)
    df_channels = pd.read_csv("multi_channel_revenue_insight.csv")
    df_channels = try_parse_timestamp(df_channels)
    data["channels"] = df_channels

    # 2. Category performance (no timestamp expected)
    df_category = pd.read_csv("processed_category_performance.csv")
    df_category = try_parse_timestamp(df_category)
    data["category"] = df_category

    # 3. Hourly sales (timestamp REQUIRED)
    df_hourly = pd.read_csv("processed_hourly_sales.csv")
    df_hourly = try_parse_timestamp(df_hourly)
    data["hourly"] = df_hourly

    # 4. Item performance (no timestamp expected)
    df_items = pd.read_csv("processed_item_performance.csv")
    df_items = try_parse_timestamp(df_items)
    data["items"] = df_items

    # 5. Scenario modelling base (may or may not have timestamp)
    df_scenario = pd.read_csv("scenario_modelling_base.csv")
    df_scenario = try_parse_timestamp(df_scenario)
    data["scenario"] = df_scenario

    return data


# ---------------------------------------------------------
# Example usage
# ---------------------------------------------------------
datasets = load_all_data()

print("Loaded datasets:")
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Loaded datasets:
channels: (4, 3)
category: (8, 5)
hourly: (30, 3)
items: (63, 7)
scenario: (1, 4)


# Next step below builds the forecasting pipeline using hourly dataset

In [36]:
# use raw annoymised restaurant sales to build forecasting time-series and produce forecasting timeseries csv
# Step 1: Imports
import pandas as pd
import numpy as np

from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

# Step 2: Build a clean time-series from the raw csv
def load_raw_data(path=r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Anonymized_Restaurant_Sales_Data.csv"):
    df = pd.read_csv(path)

    # Standardise column names if needed
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]

    # Optional: filter out cancelled orders
    if "Cancelled" in df.columns:
        df = df[df["Cancelled"].astype(str).str.lower().isin(["no", "0", "false"])]

    return df


def build_timeseries(df):
    # Clean column names
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]

    # Parse Date
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    # Parse Time safely (handles HH:MM or HH:MM:SS)
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce").dt.time

    # Drop rows where date or time failed to parse
    df = df.dropna(subset=["Date", "Time"])

    # Combine Date + Time into timestamp
    df["timestamp"] = df.apply(
        lambda r: pd.Timestamp.combine(r["Date"], r["Time"]),
        axis=1
    )

    # Aggregate to hourly level
    df["timestamp"] = df["timestamp"].dt.floor("H")

    df = df.groupby("timestamp", as_index=False)["Gross_Sales"].sum()
    df = df.rename(columns={"Gross_Sales": "revenue"}).sort_values("timestamp")

    return df


raw_df = load_raw_data()
ts_df = build_timeseries(raw_df)

print(ts_df.head())

# Step 3: Prophet forecasting
def prophet_forecast(ts_df, periods=30, freq="H"):
    df_p = ts_df.rename(columns={"timestamp": "ds", "revenue": "y"})

    m = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=True,
        daily_seasonality=True,
        seasonality_mode="additive"
    )
    m.fit(df_p)

    future = m.make_future_dataframe(periods=periods, freq=freq)
    forecast = m.predict(future)

    return forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]

# Step 4: ARIMA forecasting
def arima_forecast(ts_df, order=(2, 1, 2), steps=30):
    ts = ts_df.set_index("timestamp")["revenue"].asfreq("H").fillna(0)

    model = ARIMA(ts, order=order)
    model_fit = model.fit()

    fc = model_fit.forecast(steps=steps)
    fc = fc.rename("arima_forecast").to_frame().reset_index()

    return fc

# Step 5: Seasonality decomposition
def decompose_series(ts_df, model="additive", period=24):
    ts = ts_df.set_index("timestamp")["revenue"].asfreq("H").fillna(0)
    decomp = seasonal_decompose(ts, model=model, period=period)
    return decomp

# Step 6: What-If scenario forecasting
def scenario_forecast(prophet_fc, uplift_pct=0.10):
    df = prophet_fc.copy()
    df["scenario_yhat"] = df["yhat"] * (1 + uplift_pct)
    return df

# Step 7: Run the full forecasting pipeline
def run_forecasting_pipeline(raw_path=r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Anonymized_Restaurant_Sales_Data.csv"):
    raw = load_raw_data(raw_path)
    ts_df = build_timeseries(raw)

    prophet_fc = prophet_forecast(ts_df, periods=30, freq="H")
    arima_fc = arima_forecast(ts_df, steps=30)
    decomp = decompose_series(ts_df, period=24)

    scenario_fc = scenario_forecast(prophet_fc, uplift_pct=0.10)

    return {
        "timeseries": ts_df,
        "prophet_forecast": prophet_fc,
        "arima_forecast": arima_fc,
        "scenario_forecast": scenario_fc,
        "trend": decomp.trend,
        "seasonal": decomp.seasonal,
        "resid": decomp.resid
    }


results = run_forecasting_pipeline()
print(results["prophet_forecast"].head())
print(results["scenario_forecast"].head())

# save forecasting dataset
def save_forecast_csv(df, filename="forecast_output.csv"):
    df.to_csv(filename, index=False)
    print(f"Saved: {filename}")

# export the csv
def run_forecasting_pipeline(raw_path=r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Anonymized_Restaurant_Sales_Data.csv"):
    raw = load_raw_data(raw_path)
    ts_df = build_timeseries(raw)

    prophet_fc = prophet_forecast(ts_df, periods=30, freq="h")
    arima_fc = arima_forecast(ts_df, steps=30)
    decomp = decompose_series(ts_df, period=24)

    scenario_fc = scenario_forecast(prophet_fc, uplift_pct=0.10)

    # Merge Prophet + ARIMA + Scenario into one CSV
    merged = prophet_fc.copy()
    merged["arima"] = arima_fc["arima_forecast"]
    merged["scenario_yhat"] = scenario_fc["scenario_yhat"]

    save_forecast_csv(merged, "forecast_output.csv")

    return {
        "timeseries": ts_df,
        "prophet_forecast": prophet_fc,
        "arima_forecast": arima_fc,
        "scenario_forecast": scenario_fc,
        "merged_forecast": merged,
        "trend": decomp.trend,
        "seasonal": decomp.seasonal,
        "resid": decomp.resid
    }

results = run_forecasting_pipeline()
results["merged_forecast"].head()

C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:32: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Time"] = pd.to_datetime(df["Time"], errors="coerce").dt.time
C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["timestamp"] = df.apply(
C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:44: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["timestamp"] = df["timestamp"].dt.floor("H")
C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:44: SettingWith

            timestamp  revenue
0 2023-01-02 17:00:00     60.0
1 2023-01-02 20:00:00     62.0
2 2023-01-03 16:00:00     19.0
3 2023-01-03 18:00:00     12.0
4 2023-01-03 19:00:00     21.5


C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["timestamp"] = df.apply(
C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:44: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df["timestamp"] = df["timestamp"].dt.floor("H")
C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["timestamp"] = df["timestamp"].dt.flo

                   ds       yhat  yhat_lower  yhat_upper
0 2023-01-02 17:00:00  40.258951    8.745011   72.853539
1 2023-01-02 20:00:00  30.484598   -1.996977   61.430988
2 2023-01-03 16:00:00  34.202799    2.495049   61.707177
3 2023-01-03 18:00:00  37.767660    6.136575   67.955100
4 2023-01-03 19:00:00  30.024664   -0.116168   61.106898
                   ds       yhat  yhat_lower  yhat_upper  scenario_yhat
0 2023-01-02 17:00:00  40.258951    8.745011   72.853539      44.284846
1 2023-01-02 20:00:00  30.484598   -1.996977   61.430988      33.533058
2 2023-01-03 16:00:00  34.202799    2.495049   61.707177      37.623079
3 2023-01-03 18:00:00  37.767660    6.136575   67.955100      41.544425
4 2023-01-03 19:00:00  30.024664   -0.116168   61.106898      33.027130


09:38:35 - cmdstanpy - INFO - Chain [1] start processing
09:38:36 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:76: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ts = ts_df.set_index("timestamp")["revenue"].asfreq("H").fillna(0)
C:\Users\omota\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\omota\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


Saved: forecast_output.csv


C:\Users\omota\AppData\Local\Temp\ipykernel_35740\4255535666.py:88: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ts = ts_df.set_index("timestamp")["revenue"].asfreq("H").fillna(0)


,ds,yhat,yhat_lower,yhat_upper,arima,scenario_yhat
0,2023-01-02 17:00:00,40.258951,10.959961,71.406745,15.346312,44.284846
1,2023-01-02 20:00:00,30.484598,-0.849954,59.276845,8.864732,33.533058
2,2023-01-03 16:00:00,34.202799,3.336732,65.079489,4.896805,37.623079
3,2023-01-03 18:00:00,37.767660,8.683908,67.269789,2.988814,41.544425
4,2023-01-03 19:00:00,30.024664,-0.782434,59.701454,2.060381,33.027130


# Integrate Forecasting Into LLM Insight Engine with future-focused insights

In [46]:
# Step 1: Imports
import pandas as pd
import numpy as np
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose

# Step 2: Load Raw Data + Build Time Series
def load_raw_data(path=r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Anonymized_Restaurant_Sales_Data.csv"):
    df = pd.read_csv(path)
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]

    if "Cancelled" in df.columns:
        df = df[df["Cancelled"].astype(str).str.lower().isin(["no", "0", "false"])]

    return df


def build_timeseries(df):
    df = df.copy()

    # Detect revenue column
    revenue_col = next((c for c in df.columns if "gross" in c.lower() and "sale" in c.lower()), None)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

    # Robust Time parsing
    df["Time"] = (
        df["Time"]
        .astype(str)
        .str.strip()
        .replace("", np.nan)
    )
    df["Time"] = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce").dt.time

    df = df.dropna(subset=["Date", "Time"])

    df["timestamp"] = df.apply(lambda r: pd.Timestamp.combine(r["Date"], r["Time"]), axis=1)
    df["timestamp"] = df["timestamp"].dt.floor("h")

    df = df.groupby("timestamp", as_index=False)[revenue_col].sum()
    df = df.rename(columns={revenue_col: "revenue"})

    return df


# Step 3: Forecasting Pipeline
def prophet_forecast(ts_df, periods=24*7, freq="H"):
    df_p = ts_df.rename(columns={"timestamp": "ds", "revenue": "y"})
    m = Prophet(weekly_seasonality=True, daily_seasonality=True)
    m.fit(df_p)

    future = m.make_future_dataframe(periods=periods, freq=freq)
    forecast = m.predict(future)

    return forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]
    

def arima_forecast(ts_df, steps=30):
    ts = ts_df.set_index("timestamp")["revenue"].asfreq("h").fillna(0)
    model = ARIMA(ts, order=(2,1,2))
    model_fit = model.fit()
    fc = model_fit.forecast(steps=steps)
    return fc.rename("arima_forecast").to_frame().reset_index()


def decompose_series(ts_df, period=24):
    ts = ts_df.set_index("timestamp")["revenue"].asfreq("h").fillna(0)
    return seasonal_decompose(ts, model="additive", period=period)


def run_forecasting_pipeline():
    raw = load_raw_data()
    ts_df = build_timeseries(raw)

    prophet_fc = prophet_forecast(ts_df)
    arima_fc = arima_forecast(ts_df)
    decomp = decompose_series(ts_df)

    return {
        "timeseries": ts_df,
        "prophet_forecast": prophet_fc,
        "arima_forecast": arima_fc,
        "trend": decomp.trend,
        "seasonal": decomp.seasonal,
        "resid": decomp.resid
    }

# Step 4: Metrics and Descriptive LLM
def generate_insights(metrics, llm_fn):
    prompt = "Generate insights"
    return {"llm_output": llm_fn(prompt)}

def llm(prompt, metrics=None):

    # FORECAST INSIGHTS
    if "Latest forecast" in prompt:
        return f"""
FORECAST INSIGHTS:
- Expected revenue next hour: £{metrics['forecast']['yhat']:.2f}
- Expected revenue tomorrow (open hours only): £{metrics['daily_forecast']['next_day_revenue']:.2f}
- Confidence interval: £{metrics['forecast']['yhat_lower']:.2f} to £{metrics['forecast']['yhat_upper']:.2f}
- Trend suggests continued growth.
"""

    # MULTI-CHANNEL FORECAST INSIGHTS
    if "Multi-channel forecast" in prompt:
        mc = metrics["multi_channel_forecast"]

        latest_date = mc["latest_date"]
        total_yhat = mc["total_yhat"]
        breakdown = mc["channel_breakdown"]

        breakdown_text = "\n".join([f"- {ch}: £{val:.2f}" for ch, val in breakdown.items()])

        return f"""
MULTI-CHANNEL FORECAST INSIGHTS:
- Forecast month: {latest_date.strftime('%B %Y')}
- Total expected revenue: £{total_yhat:.2f}

Channel breakdown:
{breakdown_text}

Interpretation:
- Third-party platforms show stable monthly patterns.
- Channel mix suggests predictable customer behaviour.
- Opportunities exist to grow weaker channels through targeted promotions.
- Risks include over-reliance on a single platform or seasonal dips.
"""

    # SCENARIO INSIGHTS
    if "Scenario modelling" in prompt:
        return f"""
SCENARIO INSIGHTS:
- Scenario revenue: £{metrics['scenario_calc']['scenario_revenue']:.2f}
- Scenario profit: £{metrics['scenario_calc']['scenario_profit']:.2f}
- This scenario improves profitability and reduces risk exposure.
- Recommended: test this scenario over a 7‑day period before full rollout.
"""

    # DEFAULT DESCRIPTIVE INSIGHTS
    cat = metrics["category"]
    ops = metrics["operations"]
    items = metrics["items"]
    sc = metrics["scenario"]
    ch = metrics["channels"]

    return f"""
SUMMARY:
The business shows strong performance in the '{cat['top_category']}' category, generating £{cat['top_category_revenue']:.2f}, 
while '{cat['worst_category']}' is significantly underperforming. Operational demand peaks at {ops['peak_hour']}:00, 
with the slowest period at {ops['slowest_hour']}:00. Item-level profitability is consistent, with margins ranging 
from {items['lowest_margin_value']:.1f}% to {items['highest_margin_value']:.1f}%. The break-even point is £{sc['break_even_point']:.2f}, 
driven by fixed costs of £{sc['fixed_costs']:.2f}. EPOS (In-Store) remains the dominant revenue channel.

KEY INSIGHTS:
- '{cat['top_category']}' is the strongest category, outperforming '{cat['worst_category']}' by a wide margin.
- Peak demand at {ops['peak_hour']}:00 suggests earlier customer behaviour than typical restaurants.
- '{items['highest_margin_item']}' delivers the highest margin, while '{items['lowest_margin_item']}' is the weakest.
- Break-even point represents {sc['break_even_point'] / sc['total_revenue'] * 100:.1f}% of total revenue.
- The strongest channel is {max(ch['channel_mix'], key=ch['channel_mix'].get)} at {max(ch['channel_mix'].values()):.1f}% of total revenue.


ANOMALIES:
- Evening sales are unusually weak.
- Revenue dependency on EPOS is high.
- Category margins are unusually consistent.

RECOMMENDATIONS:
1. Promote high-margin items.
2. Review pricing for low-margin items.
3. Strengthen weaker channels.
4. Introduce evening promotions.
5. Reassess fixed costs.

RISKS & OPPORTUNITIES:
- Risk: Over-reliance on EPOS.
- Opportunity: Website and Deliveroo growth.
- Opportunity: Menu optimisation.
"""

# Step 5: Forecast + Scenario LLM Modules
def generate_llm_forecast_prompt(forecast_df):
    latest = forecast_df.iloc[-1]
    return f"""
    You are an AI business analyst.

    Latest forecast:
    - Timestamp: {latest['ds']}
    - Expected revenue: £{latest['yhat']:.2f}
    - Lower bound: £{latest['yhat_lower']:.2f}
    - Upper bound: £{latest['yhat_upper']:.2f}

    Provide:
    - A forward-looking narrative
    - Risks
    - Opportunities
    - Recommendations
    """


def scenario_engine(base_df, price_change=0, cost_change=0, volume_change=0):
    df = base_df.copy()
    df["scenario_revenue"] = df["yhat"] * (1 + price_change) * (1 + volume_change)
    df["scenario_profit"] = df["scenario_revenue"] * (1 - cost_change)
    return df


def generate_llm_scenario_prompt(scenario_df):
    latest = scenario_df.iloc[-1]
    return f"""
    Scenario modelling results:

    - Scenario revenue: £{latest['scenario_revenue']:.2f}
    - Scenario profit: £{latest['scenario_profit']:.2f}

    Provide:
    - Scenario narrative
    - Risks
    - Opportunities
    - Recommendations
    """


def generate_agentic_insights(forecast_df, scenario_df, llm_fn):
    forecast_prompt = generate_llm_forecast_prompt(forecast_df)
    scenario_prompt = generate_llm_scenario_prompt(scenario_df)

    return {
        "forecast_insights": llm_fn(forecast_prompt),
        "scenario_insights": llm_fn(scenario_prompt)
    }

# === MULTI-CHANNEL MONTHLY TIME SERIES ===

def load_monthly_multichannel(path=r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Public_DigitalChannels_Monthly_Revenue_FinancialYear.csv"):
    df = pd.read_csv(path)
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]

    # Expecting: Financial_Yr, Month_Num, Channel, Revenue
    df["Month_Num"] = df["Month_Num"].astype(int)
    df["Start_Year"] = df["Financial_Year"].astype(str).str[:4].astype(int)
    df["ds"] = pd.to_datetime(df["Start_Year"].astype(str) + "-" + df["Month_Num"].astype(str) + "-01")

    df = df[["ds", "Channel", "Revenue"]].rename(columns={"Revenue": "y"})

    return df
    
def build_channel_timeseries(df):
    channels = df["Channel"].unique().tolist()
    channel_ts = {}

    for ch in channels:
        sub = df[df["Channel"] == ch][["ds", "y"]].sort_values("ds")
        channel_ts[ch] = sub

    return channel_ts


def prophet_forecast_monthly(ts_df, periods=12):
    df_p = ts_df.rename(columns={"ds": "ds", "y": "y"})
    m = Prophet(yearly_seasonality=True)
    m.fit(df_p)

    future = m.make_future_dataframe(periods=periods, freq="MS")
    fc = m.predict(future)

    return fc[["ds", "yhat", "yhat_lower", "yhat_upper"]]


def run_monthly_multichannel_forecasting(periods=12):
    df = load_monthly_multichannel()
    channel_ts = build_channel_timeseries(df)

    forecasts = {}
    for ch, ts in channel_ts.items():
        forecasts[ch] = prophet_forecast_monthly(ts, periods=periods)

        # Combine into one table
    combined = None
    for ch, fc in forecasts.items():
        fc = fc[["ds", "yhat"]].rename(columns={"yhat": ch})
        combined = fc if combined is None else combined.merge(fc, on="ds", how="outer")

    # Prevent negative channel forecasts
    channel_cols = [c for c in combined.columns if c not in ["ds"]]
    combined[channel_cols] = combined[channel_cols].clip(lower=0)

    # Recalculate total after clamping
    combined["Total_Revenue_Forecast"] = combined[channel_cols].sum(axis=1)

    return combined, forecasts


def monthly_channel_scenario(df, price_change=None, volume_change=None):
    df = df.copy()

    if price_change is None:
        price_change = {}
    if volume_change is None:
        volume_change = {}

    channel_cols = [c for c in df.columns if c not in ["ds", "Total_Revenue_Forecast"]]

    for ch in channel_cols:
        p = price_change.get(ch, 0)
        v = volume_change.get(ch, 0)
        df[f"{ch}_scenario"] = df[ch] * (1 + p) * (1 + v)

    scenario_cols = [c for c in df.columns if c.endswith("_scenario")]
    df["Total_Scenario_Revenue"] = df[scenario_cols].sum(axis=1)

    return df


# Step 6: Run
# 1. Extract metrics
def extract_metrics():
    # Load processed CSVs
    cat_df = pd.read_csv("processed_category_performance.csv")
    ops_df = pd.read_csv("processed_hourly_sales.csv")
    item_df = pd.read_csv("processed_item_performance.csv")
    scenario_df = pd.read_csv("scenario_modelling_base.csv")

    # --- CATEGORY METRICS ---
    top_cat = cat_df.loc[cat_df["Gross Sales"].idxmax()]
    worst_cat = cat_df.loc[cat_df["Gross Sales"].idxmin()]

    category_metrics = {
        "top_category": top_cat["Category"],
        "top_category_revenue": top_cat["Gross Sales"],
        "worst_category": worst_cat["Category"],
        "worst_category_revenue": worst_cat["Gross Sales"]
    }

    # --- OPERATIONS METRICS ---
    peak_hour_row = ops_df.loc[ops_df["Gross Sales"].idxmax()]
    slowest_hour_row = ops_df.loc[ops_df["Gross Sales"].idxmin()]

    operations_metrics = {
        "peak_hour": int(peak_hour_row["Hour"]),
        "slowest_hour": int(slowest_hour_row["Hour"])
    }

    # --- ITEM METRICS ---
    highest_margin = item_df.loc[item_df["Margin_Percentage"].idxmax()]
    lowest_margin = item_df.loc[item_df["Margin_Percentage"].idxmin()]

    item_metrics = {
        "highest_margin_item": highest_margin["Line item name"],
        "highest_margin_value": highest_margin["Margin_Percentage"],
        "lowest_margin_item": lowest_margin["Line item name"],
        "lowest_margin_value": lowest_margin["Margin_Percentage"]
    }

    # --- SCENARIO METRICS ---
    scenario_metrics = {
        "break_even_point": scenario_df["Break_Even_Point"].iloc[0],
        "fixed_costs": scenario_df["Fixed_Costs"].iloc[0],
        "total_revenue": scenario_df["Total_Revenue"].iloc[0]
    }

    # --- CHANNEL METRICS (from monthly multichannel) ---
    mc_df = load_monthly_multichannel()
    channel_mix = (
        mc_df.groupby("Channel")["y"].sum()
        / mc_df["y"].sum()
        * 100
    ).to_dict()

    channels_metrics = {
        "channel_mix": channel_mix,
        "lowest_channel": min(channel_mix, key=channel_mix.get)
    }

    return {
        "category": category_metrics,
        "operations": operations_metrics,
        "items": item_metrics,
        "scenario": scenario_metrics,
        "channels": channels_metrics
    }

# MAIN EXECUTION

metrics = extract_metrics()

# 2. Descriptive insights
descriptive = generate_insights(metrics, lambda p: llm(p, metrics))
print("\n=== DESCRIPTIVE INSIGHTS ===\n")
print(descriptive["llm_output"])

# 3. Forecasting
results = run_forecasting_pipeline()
forecast_df = results["prophet_forecast"]

# 3b. Multi-channel monthly forecasting
combined, forecasts = run_monthly_multichannel_forecasting(periods=12)

latest_row = combined.iloc[-1]
metrics["multi_channel_forecast"] = {
    "latest_date": latest_row["ds"],
    "total_yhat": latest_row["Total_Revenue_Forecast"],
    "channel_breakdown": {ch: latest_row[ch] for ch in forecasts.keys()}
}

mc_prompt = "Multi-channel forecast: provide narrative, risks, opportunities, and recommendations."
mc_insights = llm(mc_prompt, metrics)
print("\n=== MULTI-CHANNEL FORECAST INSIGHTS ===\n")
print(mc_insights)

# 4a. DAILY FORECAST (opening hours only, FUTURE ONLY)
forecast_df["ds"] = pd.to_datetime(forecast_df["ds"])
forecast_df["hour"] = forecast_df["ds"].dt.hour

last_history_ts = results["timeseries"]["timestamp"].max()
future_fc = forecast_df[forecast_df["ds"] > last_history_ts]
open_hours_forecast = future_fc[future_fc["hour"].between(16, 21)]

daily_forecast = open_hours_forecast.set_index("ds").resample("D")["yhat"].sum().reset_index()

metrics["daily_forecast"] = {
    "next_day_revenue": daily_forecast.iloc[0]["yhat"]
}

# 4b. Scenario values into metrics for the LLM
scenario_df = scenario_engine(forecast_df)
metrics["forecast"] = {
    "yhat": forecast_df.iloc[-1]["yhat"],
    "yhat_lower": forecast_df.iloc[-1]["yhat_lower"],
    "yhat_upper": forecast_df.iloc[-1]["yhat_upper"]
}
metrics["scenario_calc"] = {
    "scenario_revenue": scenario_df.iloc[-1]["scenario_revenue"],
    "scenario_profit": scenario_df.iloc[-1]["scenario_profit"]
}

print("\n=== DAILY FORECAST (OPENING HOURS ONLY) ===\n")
print(daily_forecast.head())

# 5. Agentic insights
agentic = generate_agentic_insights(forecast_df, scenario_df, lambda p: llm(p, metrics))

print("\n=== FORECAST INSIGHTS ===\n")
print(agentic["forecast_insights"])

print("\n=== SCENARIO INSIGHTS ===\n")
print(agentic["scenario_insights"])



=== DESCRIPTIVE INSIGHTS ===


SUMMARY:
The business shows strong performance in the 'EXTRAS' category, generating £12777.50, 
while 'Friday Specials' is significantly underperforming. Operational demand peaks at 16:00, 
with the slowest period at 20:00. Item-level profitability is consistent, with margins ranging 
from 64.9% to 65.1%. The break-even point is £214895.36, 
driven by fixed costs of £164400.40. EPOS (In-Store) remains the dominant revenue channel.

KEY INSIGHTS:
- 'EXTRAS' is the strongest category, outperforming 'Friday Specials' by a wide margin.
- Peak demand at 16:00 suggests earlier customer behaviour than typical restaurants.
- 'Chin Chin' delivers the highest margin, while 'Maltina' is the weakest.
- Break-even point represents 74.8% of total revenue.
- The strongest channel is JustEat at 48.2% of total revenue.


ANOMALIES:
- Evening sales are unusually weak.
- Revenue dependency on EPOS is high.
- Category margins are unusually consistent.

RECOMMENDATIONS:
1. P

17:17:50 - cmdstanpy - INFO - Chain [1] start processing
17:17:53 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\omota\anaconda3\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(
C:\Users\omota\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\omota\anaconda3\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
17:18:05 - cmdstanpy - INFO - Chain [1] start processing
17:18:05 - cmdstanpy - INFO - Chain [1] done processing
17:18:05 - cmdstanpy - INFO - Chain [1] start processing
17:18:05 - cmdstanpy - INFO - Chain [1


=== MULTI-CHANNEL FORECAST INSIGHTS ===


MULTI-CHANNEL FORECAST INSIGHTS:
- Forecast month: December 2026
- Total expected revenue: £2438.33

Channel breakdown:
- Deliveroo: £790.30
- JustEat: £1648.03
- Website: £0.00

Interpretation:
- Third-party platforms show stable monthly patterns.
- Channel mix suggests predictable customer behaviour.
- Opportunities exist to grow weaker channels through targeted promotions.
- Risks include over-reliance on a single platform or seasonal dips.


=== DAILY FORECAST (OPENING HOURS ONLY) ===

          ds        yhat
0 2025-12-31   78.932701
1 2026-01-01  193.243616
2 2026-01-02  190.604560
3 2026-01-03  219.544845
4 2026-01-04  166.615736

=== FORECAST INSIGHTS ===


FORECAST INSIGHTS:
- Expected revenue next hour: £34.52
- Expected revenue tomorrow (open hours only): £78.93
- Confidence interval: £8.31 to £61.66
- Trend suggests continued growth.


=== SCENARIO INSIGHTS ===


SCENARIO INSIGHTS:
- Scenario revenue: £34.52
- Scenario profit: £34.

# Epos weekly + monthly forecasting module

In [48]:
"""
===========================================================
EPOS WEEKLY + MONTHLY FORECASTING MODULE
===========================================================

Purpose:
--------
This script extends the EPOS forecasting pipeline by adding:

1. Weekly EPOS revenue forecast
2. Monthly EPOS revenue forecast
3. Optional: Combined EPOS + Digital Channels monthly forecast

Inputs:
-------
- Raw EPOS sales data (hourly timestamps)
- Prophet forecast output from your existing EPOS model

Outputs:
--------
- Weekly EPOS revenue forecast
- Monthly EPOS revenue forecast
- (Optional) Combined EPOS + Digital Channels monthly forecast

===========================================================
"""

# -----------------------------
# Imports
# -----------------------------
import pandas as pd
import numpy as np
from prophet import Prophet


# -----------------------------
# Load Raw EPOS Data
# -----------------------------
def load_raw_epos(path=r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Anonymized_Restaurant_Sales_Data.csv"):
    df = pd.read_csv(path)
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]

    # Remove cancelled orders if present
    if "Cancelled" in df.columns:
        df = df[df["Cancelled"].astype(str).str.lower().isin(["no", "0", "false"])]

    return df


# -----------------------------
# Build Hourly Time Series
# -----------------------------
def build_epos_timeseries(df):
    df = df.copy()

    # Detect revenue column
    revenue_col = next((c for c in df.columns if "gross" in c.lower() and "sale" in c.lower()), None)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)

    # Parse time safely
    df["Time"] = (
        df["Time"]
        .astype(str)
        .str.strip()
        .replace("", np.nan)
    )
    df["Time"] = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce").dt.time

    df = df.dropna(subset=["Date", "Time"])

    df["timestamp"] = df.apply(lambda r: pd.Timestamp.combine(r["Date"], r["Time"]), axis=1)
    df["timestamp"] = df["timestamp"].dt.floor("h")

    df = df.groupby("timestamp", as_index=False)[revenue_col].sum()
    df = df.rename(columns={revenue_col: "revenue"})

    return df


# -----------------------------
# Prophet Hourly Forecast
# -----------------------------
def prophet_hourly_forecast(ts_df, periods=24*7, freq="H"):
    df_p = ts_df.rename(columns={"timestamp": "ds", "revenue": "y"})
    m = Prophet(weekly_seasonality=True, daily_seasonality=True)
    m.fit(df_p)

    future = m.make_future_dataframe(periods=periods, freq=freq)
    forecast = m.predict(future)

    return forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]]


# -----------------------------
# Weekly EPOS Forecast
# -----------------------------
def build_weekly_epos_forecast(forecast_df, ts_df):
    last_history_ts = ts_df["timestamp"].max()

    # Only future predictions
    df = forecast_df[forecast_df["ds"] > last_history_ts].copy()
    df["ds"] = pd.to_datetime(df["ds"])
    df["hour"] = df["ds"].dt.hour
    df["day"] = df["ds"].dt.day_name()

    # Filter to opening hours only: Mon–Sat, 16:00–21:00
    df = df[
        (df["hour"].between(16, 21)) &
        (df["day"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]))
    ]

    df = df.set_index("ds")
    weekly = df["yhat"].resample("W-MON").sum().reset_index()
    weekly = weekly.rename(columns={"yhat": "weekly_revenue"})
    return weekly


# -----------------------------
# Monthly EPOS Forecast
# -----------------------------
def build_monthly_epos_forecast(forecast_df, ts_df):
    last_history_ts = ts_df["timestamp"].max()

    df = forecast_df[forecast_df["ds"] > last_history_ts].copy()
    df["ds"] = pd.to_datetime(df["ds"])
    df["hour"] = df["ds"].dt.hour
    df["day"] = df["ds"].dt.day_name()

    # Filter to opening hours only: Mon–Sat, 16:00–21:00
    df = df[
        (df["hour"].between(16, 21)) &
        (df["day"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]))
    ]

    df = df.set_index("ds")
    monthly = df["yhat"].resample("MS").sum().reset_index()
    monthly = monthly.rename(columns={"yhat": "monthly_revenue"})
    return monthly


# -----------------------------
# Optional: Combine EPOS + Digital Channels
# -----------------------------

def combine_epos_and_digital(monthly_epos, digital_df):
    """
    Combine EPOS monthly forecast with digital channel monthly forecast.

    monthly_epos: DataFrame with ['ds', 'monthly_revenue']
    digital_df: DataFrame with ['ds', 'Deliveroo', 'JustEat', 'Website', 'Total_Revenue_Forecast']
    """

    combined = digital_df.merge(monthly_epos, on="ds", how="left")

    # Fill missing EPOS values with 0
    combined["monthly_revenue"] = combined["monthly_revenue"].fillna(0)

    # Compute grand total
    combined["Grand_Total_Revenue"] = (
        combined["Total_Revenue_Forecast"] + combined["monthly_revenue"]
    )

    return combined


# -----------------------------
# MAIN EXECUTION
# -----------------------------
if __name__ == "__main__":

    # 1. Load EPOS raw data
    epos_raw_path = r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Anonymized_Restaurant_Sales_Data.csv"
    raw = load_raw_epos(epos_raw_path)

    # 2. Build hourly EPOS time series
    ts_df = build_epos_timeseries(raw)

    # 3. Prophet hourly forecast
    # Forecast ~1 year of hourly data (8760 hours)
    forecast_df = prophet_hourly_forecast(ts_df, periods=24*365)

    # 4. Weekly forecast
    weekly_epos = build_weekly_epos_forecast(forecast_df, ts_df)
    print("\n=== WEEKLY EPOS FORECAST ===\n")
    print(weekly_epos.head())

    # 5. Monthly forecast
    monthly_epos = build_monthly_epos_forecast(forecast_df, ts_df)
    print("\n=== MONTHLY EPOS FORECAST ===\n")
    print(monthly_epos.head())

    # 6. Digital channels monthly forecast
    # (Make sure the cell defining run_monthly_multichannel_forecasting() has been run)
    combined_digital, forecasts = run_monthly_multichannel_forecasting(periods=12)

    # 7. Combine EPOS + Digital
    combined_total = combine_epos_and_digital(monthly_epos, combined_digital)

    print("\n=== GRAND TOTAL MONTHLY FORECAST (EPOS + DIGITAL CHANNELS) ===\n")
    print(combined_total.tail())


17:18:11 - cmdstanpy - INFO - Chain [1] start processing
17:18:11 - cmdstanpy - INFO - Chain [1] done processing
C:\Users\omota\anaconda3\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(
17:18:15 - cmdstanpy - INFO - Chain [1] start processing



=== WEEKLY EPOS FORECAST ===

          ds  weekly_revenue
0 2026-01-05      879.740998
1 2026-01-12     1249.876094
2 2026-01-19     1308.137692
3 2026-01-26     1315.854877
4 2026-02-02     1281.868373

=== MONTHLY EPOS FORECAST ===

          ds  monthly_revenue
0 2025-12-01        78.932701
1 2026-01-01      5750.331829
2 2026-02-01      4203.165790
3 2026-03-01      5046.568059
4 2026-04-01      4730.312969


17:18:15 - cmdstanpy - INFO - Chain [1] done processing
17:18:16 - cmdstanpy - INFO - Chain [1] start processing
17:18:16 - cmdstanpy - INFO - Chain [1] done processing
17:18:16 - cmdstanpy - INFO - Chain [1] start processing
17:18:16 - cmdstanpy - INFO - Chain [1] done processing



=== GRAND TOTAL MONTHLY FORECAST (EPOS + DIGITAL CHANNELS) ===

           ds    Deliveroo      JustEat    Website  Total_Revenue_Forecast  \
43 2026-08-01   664.830754  1493.525653   0.000000             2158.356407   
44 2026-09-01   784.856731  1643.037187   0.000000             2427.893919   
45 2026-10-01  1051.745461  1717.862994  29.882218             2799.490673   
46 2026-11-01   718.383272  1496.081916   0.000000             2214.465188   
47 2026-12-01   790.303478  1648.027874   0.000000             2438.331352   

    monthly_revenue  Grand_Total_Revenue  
43      4397.367964          6555.724371  
44      5529.357076          7957.250995  
45      5136.838199          7936.328872  
46      4731.754543          6946.219731  
47      4499.777998          6938.109349  


# Build Category Forecasting Engine

In [ ]:
!pip install fpdf

# Redesigning the forecasting engine and the plots codes

In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
import os
from fpdf import FPDF

# -----------------------------
# CLEAN CATEGORY NAME
# -----------------------------
def clean_category_name(cat):
    if pd.isna(cat):
        return cat

    # Standard cleaning
    cat = str(cat).strip().upper().replace(" ", "_")

    # Normalise known variations
    corrections = {
        "APETISERS": "APPETISERS",
        "APPETIZER": "APPETISERS",
        "APPETISER": "APPETISERS",
        "STARTERS": "APPETISERS"  # optional
    }

    return corrections.get(cat, cat)


# -----------------------------
# LOAD RAW EPOS
# -----------------------------
def load_raw_epos(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    if "Cancelled" in df.columns:
        df = df[df["Cancelled"].astype(str).str.lower().isin(["no", "0", "false"])]
    return df

# -----------------------------
# EPOS CATEGORY MONTHLY
# -----------------------------
def build_epos_category_monthly(raw_epos):
    df = raw_epos.copy()
    df["Category"] = df["Category"].apply(clean_category_name)

    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    df["Time"] = pd.to_datetime(df["Time"], format="%H:%M", errors="coerce").dt.time
    df = df.dropna(subset=["Date", "Time"])

    df["timestamp"] = df.apply(lambda r: pd.Timestamp.combine(r["Date"], r["Time"]), axis=1)
    df["hour"] = df["timestamp"].dt.hour
    df["day"] = df["timestamp"].dt.day_name()

    df = df[
        (df["hour"].between(16, 21)) &
        (df["day"].isin(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]))
    ]

    df["ds"] = df["timestamp"].dt.to_period("M").dt.to_timestamp()

    return (
        df.groupby(["ds", "Category"], as_index=False)
        .agg({
            "Quantity": "sum",
            "Gross_Sales": "sum",
            "Est._Cost": "sum",
            "Est._Profit": "sum"
        })
        .rename(columns={
            "Quantity": "qty",
            "Gross_Sales": "revenue",
            "Est._Cost": "cost",
            "Est._Profit": "profit"
        })
    )

# -----------------------------
# DELIVEROO CATEGORY MONTHLY
# -----------------------------
def load_deliveroo_category_monthly(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    df["Category"] = df["Category"].apply(clean_category_name)

    df["ds"] = pd.to_datetime(df["Year"].astype(str) + "-" + df["Month"].astype(str) + "-01")

    return (
        df.groupby(["ds", "Category"], as_index=False)
        .agg({"Quantity": "sum", "Gross_Revenue": "sum"})
        .rename(columns={"Quantity": "qty", "Gross_Revenue": "revenue"})
    )

# -----------------------------
# PROPHET FORECAST
# -----------------------------
def prophet_category_forecast(cat_df, value_col="revenue", periods=12):
    categories = cat_df["Category"].unique().tolist()
    forecasts = {}

    for cat in categories:
        sub = cat_df[cat_df["Category"] == cat][["ds", value_col]].dropna().sort_values("ds")
        if len(sub) < 3:
            continue

        df_p = sub.rename(columns={value_col: "y"})
        m = Prophet(yearly_seasonality=True)
        m.fit(df_p)

        future = m.make_future_dataframe(periods=periods, freq="MS")
        fc = m.predict(future)

        fc["yhat"] = fc["yhat"].clip(lower=0)
        fc = fc[["ds", "yhat"]]
        fc["Category"] = cat
        fc["metric"] = value_col

        forecasts[cat] = fc

    return forecasts

def combine_forecasts_to_df(forecast_dict):
    if not forecast_dict:
        return pd.DataFrame()
    return pd.concat(forecast_dict.values(), ignore_index=True)

# -----------------------------
# RUN CATEGORY FORECASTING (THE MISSING FUNCTION)
# -----------------------------
def run_category_forecasting_epos_and_deliveroo(raw_epos, deliveroo_path, periods=12):

    epos_cat = build_epos_category_monthly(raw_epos)
    del_cat = load_deliveroo_category_monthly(deliveroo_path)

    epos_rev_fc = prophet_category_forecast(epos_cat[["ds", "Category", "revenue"]], "revenue", periods)
    epos_profit_fc = prophet_category_forecast(epos_cat[["ds", "Category", "profit"]], "profit", periods)
    del_rev_fc = prophet_category_forecast(del_cat[["ds", "Category", "revenue"]], "revenue", periods)

    return {
        "epos_cat_monthly": epos_cat,
        "deliveroo_cat_monthly": del_cat,
        "epos_revenue_forecasts": epos_rev_fc,
        "epos_profit_forecasts": epos_profit_fc,
        "deliveroo_revenue_forecasts": del_rev_fc
    }

# -----------------------------
# COMBINED CATEGORY TABLE
# -----------------------------
def build_combined_category_table(cat_results):
    epos_rev = combine_forecasts_to_df(cat_results["epos_revenue_forecasts"]).rename(columns={"yhat": "epos_revenue"})
    epos_profit = combine_forecasts_to_df(cat_results["epos_profit_forecasts"]).rename(columns={"yhat": "epos_profit"})
    del_rev = combine_forecasts_to_df(cat_results["deliveroo_revenue_forecasts"]).rename(columns={"yhat": "deliveroo_revenue"})

    combined = (
        epos_rev.merge(epos_profit, on=["ds", "Category"], how="outer")
                .merge(del_rev, on=["ds", "Category"], how="outer")
    )

    combined = combined.fillna(0)
    combined["total_revenue"] = combined["epos_revenue"] + combined["deliveroo_revenue"]
    combined["total_profit"] = combined["epos_profit"]

    combined["total_revenue_month"] = combined.groupby("ds")["total_revenue"].transform("sum")
    combined["category_share"] = combined["total_revenue"] / combined["total_revenue_month"]

    return combined


In [60]:
combined_cat["Category"].unique()

array(['APPETISERS', 'DRINKS', 'EXTRAS', 'MAIN_COURSES',
       'RESTAURANT_DEALS', 'SOUPS', 'SPECIAL_ORDERS', 'FRIDAY_SPECIALS',
       'MODIFIERS', 'SOLIDS', 'PROMOTIONAL_BUNDLES'], dtype=object)

In [54]:
# Load data
epos_path = r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Anonymized_Restaurant_Sales_Data.csv"
deliveroo_path = r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Public_Deliveroo_Category_Summary_2023_2025.csv"

raw_epos = load_raw_epos(epos_path)


In [56]:
# Run forecasting once
cat_results = run_category_forecasting_epos_and_deliveroo(
    raw_epos=raw_epos,
    deliveroo_path=deliveroo_path,
    periods=12
)

combined_cat = build_combined_category_table(cat_results)

combined_cat.head()


17:19:43 - cmdstanpy - INFO - Chain [1] start processing
17:19:43 - cmdstanpy - INFO - Chain [1] done processing
17:19:43 - cmdstanpy - INFO - Chain [1] start processing
17:19:43 - cmdstanpy - INFO - Chain [1] done processing
17:19:44 - cmdstanpy - INFO - Chain [1] start processing
17:19:44 - cmdstanpy - INFO - Chain [1] done processing
17:19:44 - cmdstanpy - INFO - Chain [1] start processing
17:19:44 - cmdstanpy - INFO - Chain [1] done processing
17:19:45 - cmdstanpy - INFO - Chain [1] start processing
17:19:45 - cmdstanpy - INFO - Chain [1] done processing
17:19:45 - cmdstanpy - INFO - Chain [1] start processing
17:19:45 - cmdstanpy - INFO - Chain [1] done processing
17:19:46 - cmdstanpy - INFO - Chain [1] start processing
17:19:50 - cmdstanpy - INFO - Chain [1] done processing
17:19:50 - cmdstanpy - INFO - Chain [1] start processing
17:19:51 - cmdstanpy - INFO - Chain [1] done processing
17:19:51 - cmdstanpy - INFO - Chain [1] start processing
17:19:51 - cmdstanpy - INFO - Chain [1]

,ds,epos_revenue,Category,metric_x,epos_profit,metric_y,deliveroo_revenue,metric,total_revenue,total_profit,total_revenue_month,category_share
0,2023-01-01,209.081167,APPETISERS,revenue,135.971319,profit,0.0,0,209.081167,135.971319,2035.765028,0.102704
1,2023-01-01,24.592438,DRINKS,revenue,15.987571,profit,0.0,0,24.592438,15.987571,2035.765028,0.012080
2,2023-01-01,666.418074,EXTRAS,revenue,433.275375,profit,0.0,0,666.418074,433.275375,2035.765028,0.327355
3,2023-01-01,428.444804,MAIN_COURSES,revenue,278.570332,profit,0.0,0,428.444804,278.570332,2035.765028,0.210459
4,2023-01-01,254.000363,RESTAURANT_DEALS,revenue,165.068778,profit,0.0,0,254.000363,165.068778,2035.765028,0.124769


In [ ]:
pip install ipywidgets

In [ ]:
pip install widgetsnbextension


In [ ]:
!pip install --upgrade ipywidgets widgetsnbextension

In [62]:
# Forecasting and Insignt Generation Code
import os
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from fpdf import FPDF
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Create folder for saving plots
os.makedirs("forecast_plots", exist_ok=True)

# ============================================================
# PER‑CATEGORY PDF EXPORT
# ============================================================

def export_category_pdf(category):
    folder = "forecast_plots"

    images = sorted([
        f for f in os.listdir(folder)
        if f.startswith(category) and f.lower().endswith(".png")
    ])

    if not images:
        print(f"No saved plots found for category: {category}")
        return

    pdf = FPDF(unit="mm", format="A4")

    for img in images:
        img_path = os.path.join(folder, img)

        im = Image.open(img_path)
        width, height = im.size
        aspect = height / width

        pdf.add_page()
        pdf.set_font("Arial", size=12)
        pdf.cell(0, 10, txt=img.replace("_", " ").replace(".png", ""), ln=True)

        new_width = 190
        new_height = new_width * aspect

        pdf.image(img_path, x=10, y=25, w=new_width, h=new_height)

    output_path = f"{category}_Forecasts.pdf"
    pdf.output(output_path)
    print(f"PDF created: {output_path}")


# ============================================================
# TOP 5 CATEGORIES BY GROWTH
# ============================================================

def top_5_categories_by_growth(combined_cat):
    growth_list = []

    for cat in combined_cat["Category"].unique():
        df = combined_cat[combined_cat["Category"] == cat].sort_values("ds")
        first = df.iloc[0]["total_revenue"]
        last = df.iloc[-1]["total_revenue"]

        growth = (last - first) / first if first > 0 else 0
        growth_list.append((cat, growth))

    growth_df = pd.DataFrame(growth_list, columns=["Category", "Growth"])
    growth_df = growth_df.sort_values("Growth", ascending=False).head(5)

    # Save as PNG
    plt.figure(figsize=(8,4))
    plt.bar(growth_df["Category"], growth_df["Growth"], color="skyblue")
    plt.title("Top 5 Categories by Revenue Growth")
    plt.ylabel("Growth Rate")
    plt.xticks(rotation=45)
    plt.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig("forecast_plots/top_5_growth.png", dpi=300)
    plt.show()

    return growth_df


# ============================================================
# CATEGORY RISK SCORE
# ============================================================

def calculate_risk_score(combined_cat, category):
    df = combined_cat[combined_cat["Category"] == category].sort_values("ds")

    first = df.iloc[0]
    last = df.iloc[-1]

    decline_factor = 1 if last["total_revenue"] < first["total_revenue"] else 0
    low_profit_factor = 1 if last["epos_profit"] <= 0 else 0

    total = last["total_revenue"]
    dep = 0
    if total > 0:
        epos_share = last["epos_revenue"] / total
        del_share = last["deliveroo_revenue"] / total
        dep = 1 if max(epos_share, del_share) > 0.8 else 0

    share_drop_factor = 1 if last["category_share"] < first["category_share"] else 0

    risk = (
        decline_factor * 40 +
        low_profit_factor * 20 +
        dep * 20 +
        share_drop_factor * 20
    )

    return risk


# ============================================================
# FORECASTING ACCURACY (MAPE, RMSE, MAE)
# ============================================================

def calculate_forecast_accuracy(raw_df, forecast_df, category):
    """
    raw_df: actual EPOS monthly revenue
    forecast_df: prophet forecast output (yhat)
    """

    actual = raw_df[raw_df["Category"] == category][["ds", "revenue"]]
    forecast = forecast_df[forecast_df["Category"] == category][["ds", "yhat"]]

    merged = actual.merge(forecast, on="ds", how="inner")

    if len(merged) < 3:
        return None  # not enough data

    y_true = merged["revenue"]
    y_pred = merged["yhat"]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }


def show_accuracy_table(epos_cat, epos_forecasts):
    results = []

    for cat in epos_cat["Category"].unique():
        if cat not in epos_forecasts:
            continue

        acc = calculate_forecast_accuracy(epos_cat, epos_forecasts[cat], cat)
        if acc:
            results.append([cat, acc["MAE"], acc["RMSE"], acc["MAPE"]])

    df = pd.DataFrame(results, columns=["Category", "MAE", "RMSE", "MAPE"])
    df = df.sort_values("MAPE")

    # Save as PNG
    plt.figure(figsize=(10,4))
    plt.table(cellText=df.values, colLabels=df.columns, loc="center")
    plt.axis("off")
    plt.title("Forecast Accuracy (Lower is Better)")
    plt.savefig("forecast_plots/forecast_accuracy.png", dpi=300, bbox_inches="tight")
    plt.show()

    return df


# ============================================================
# PLOTTING FUNCTIONS (Auto-save + Show)
# ============================================================

def plot_revenue_for_category(combined_cat, category):
    df = combined_cat[combined_cat["Category"] == category].sort_values("ds")

    plt.figure(figsize=(12,6))
    plt.plot(df["ds"], df["epos_revenue"], label="EPOS Revenue", marker="o")
    plt.plot(df["ds"], df["deliveroo_revenue"], label="Deliveroo Revenue", marker="o")
    plt.title(f"Revenue Forecast: {category}")
    plt.xlabel("Month")
    plt.ylabel("Revenue (£)")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"forecast_plots/{category}_revenue_forecast.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_profit_for_category(combined_cat, category):
    df = combined_cat[combined_cat["Category"] == category].sort_values("ds")

    plt.figure(figsize=(12,6))
    plt.plot(df["ds"], df["epos_profit"], label="EPOS Profit", marker="o", color="green")
    plt.title(f"Profit Forecast: {category}")
    plt.xlabel("Month")
    plt.ylabel("Profit (£)")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"forecast_plots/{category}_profit_forecast.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_combined_for_category(combined_cat, category):
    df = combined_cat[combined_cat["Category"] == category].sort_values("ds")

    plt.figure(figsize=(12,6))
    plt.plot(df["ds"], df["epos_revenue"], label="EPOS Revenue", marker="o")
    plt.plot(df["ds"], df["deliveroo_revenue"], label="Deliveroo Revenue", marker="o")
    plt.plot(df["ds"], df["epos_profit"], label="EPOS Profit", marker="o", color="green")

    plt.title(f"Combined Forecast: {category}")
    plt.xlabel("Month")
    plt.ylabel("£ Value")
    plt.legend()
    plt.grid(True)

    plt.savefig(f"forecast_plots/{category}_combined_forecast.png", dpi=300, bbox_inches="tight")
    plt.show()


# ============================================================
# EPOS vs DELIVEROO COMPARISON (Auto-save + Show)
# ============================================================

def compare_epos_vs_deliveroo(combined_cat, category):
    df = combined_cat[combined_cat["Category"] == category].sort_values("ds")

    plt.figure(figsize=(14,8))

    plt.subplot(2, 1, 1)
    plt.plot(df["ds"], df["epos_revenue"], label="EPOS Revenue", marker="o")
    plt.plot(df["ds"], df["deliveroo_revenue"], label="Deliveroo Revenue", marker="o")
    plt.title(f"EPOS vs Deliveroo Revenue Comparison: {category}")
    plt.ylabel("Revenue (£)")
    plt.legend()
    plt.grid(True)

    plt.subplot(2, 1, 2)
    plt.plot(df["ds"], df["epos_profit"], label="EPOS Profit", marker="o", color="green")
    plt.plot(df["ds"], df["category_share"] * 100, label="Category Share (%)", marker="o", color="purple")
    plt.title("Profit & Category Share")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"forecast_plots/{category}_epos_vs_deliveroo.png", dpi=300, bbox_inches="tight")
    plt.show()


# ============================================================
# SMART INSIGHT FUNCTION
# ============================================================

def generate_category_insights(combined_cat, category):
    df = combined_cat[combined_cat["Category"] == category].sort_values("ds")

    latest = df.iloc[-1]
    earliest = df.iloc[0]

    # Margin
    margin = 0
    if latest["total_revenue"] > 0:
        margin = latest["epos_profit"] / latest["total_revenue"]

    # Risk score
    risk = calculate_risk_score(combined_cat, category)

    # Base insight
    insight = f"""
Category: {category}

• Revenue changes from £{earliest['total_revenue']:.2f} to £{latest['total_revenue']:.2f}.
• EPOS contributes £{latest['epos_revenue']:.2f}.
• Deliveroo contributes £{latest['deliveroo_revenue']:.2f}.
• Profit reaches £{latest['epos_profit']:.2f}, margin {margin:.1%}.
• Category share ends at {latest['category_share']:.1%}.
• Risk Score: {risk}/100
"""

    # --- SHORT EXECUTIVE SUMMARY INTERPRETATION (Option C) ---
    interpretation = "Interpretation (Executive Summary):\n"

    # Trend
    if latest["total_revenue"] > earliest["total_revenue"]:
        interpretation += "• Upward trend with improving demand.\n"
    elif latest["total_revenue"] < earliest["total_revenue"]:
        interpretation += "• Downward trend indicating weakening demand.\n"
    else:
        interpretation += "• Flat trend with stable demand.\n"

    # Profitability
    if latest["epos_profit"] <= 0:
        interpretation += "• Profitability is weak, increasing operational risk.\n"
    else:
        interpretation += "• Profitability is stable.\n"

    # Channel dependency
    total = latest["total_revenue"]
    if total > 0:
        epos_share = latest["epos_revenue"] / total
        del_share = latest["deliveroo_revenue"] / total

        if epos_share > 0.8:
            interpretation += "• Strong EPOS dependency.\n"
        elif del_share > 0.8:
            interpretation += "• Strong Deliveroo dependency.\n"
        else:
            interpretation += "• Balanced channel mix.\n"

    # Category share
    if latest["category_share"] < df.iloc[0]["category_share"]:
        interpretation += "• Category share is declining.\n"
    else:
        interpretation += "• Category share is stable or improving.\n"

    return insight + "\n" + interpretation


# ============================================================
# INTERACTIVE WIDGET DASHBOARD
# ============================================================

out = widgets.Output()

category_dropdown = widgets.Dropdown(
    options=sorted(combined_cat["Category"].unique()),
    description="Category:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

pdf_button = widgets.Button(
    description="Export Category PDF",
    button_style="success",
    layout=widgets.Layout(width='250px')
)

growth_button = widgets.Button(
    description="Show Top 5 Growth",
    button_style="info",
    layout=widgets.Layout(width='250px')
)

accuracy_button = widgets.Button(
    description="Show Forecast Accuracy",
    button_style="warning",
    layout=widgets.Layout(width='250px')
)

def on_pdf_button_clicked(b):
    export_category_pdf(category_dropdown.value)

def on_growth_button_clicked(b):
    with out:
        out.clear_output(wait=True)
        print(top_5_categories_by_growth(combined_cat))

def on_accuracy_button_clicked(b):
    with out:
        out.clear_output(wait=True)
        print(show_accuracy_table(
            cat_results["epos_cat_monthly"],
            cat_results["epos_revenue_forecasts"]
        ))

pdf_button.on_click(on_pdf_button_clicked)
growth_button.on_click(on_growth_button_clicked)
accuracy_button.on_click(on_accuracy_button_clicked)


def update_dashboard(change):
    with out:
        out.clear_output(wait=True)

        category = change["new"]

        plot_revenue_for_category(combined_cat, category)
        plot_profit_for_category(combined_cat, category)
        plot_combined_for_category(combined_cat, category)
        compare_epos_vs_deliveroo(combined_cat, category)

        print(generate_category_insights(combined_cat, category))


category_dropdown.observe(update_dashboard, names='value')

display(category_dropdown, pdf_button, growth_button, accuracy_button, out)

update_dashboard({'name': 'value', 'new': category_dropdown.value})


Dropdown(description='Category:', layout=Layout(width='300px'), options=('APPETISERS', 'DRINKS', 'EXTRAS', 'FR…

Button(button_style='success', description='Export Category PDF', layout=Layout(width='250px'), style=ButtonSt…

Button(button_style='info', description='Show Top 5 Growth', layout=Layout(width='250px'), style=ButtonStyle()…

Button(button_style='warning', description='Show Forecast Accuracy', layout=Layout(width='250px'), style=Butto…

Output()

In [64]:
cat_results.keys()

dict_keys(['epos_cat_monthly', 'deliveroo_cat_monthly', 'epos_revenue_forecasts', 'epos_profit_forecasts', 'deliveroo_revenue_forecasts'])

In [84]:
# Forecast Persistence Code (Saving Prophet Forecasts for Dashboard Use)
import pickle

epos_revenue_forecasts = cat_results["epos_revenue_forecasts"]

with open("epos_revenue_forecasts.pkl", "wb") as f:
    pickle.dump(epos_revenue_forecasts, f)

print("Saved: epos_revenue_forecasts.pkl")


Saved: epos_revenue_forecasts.pkl


# Build the final data loading script to load all 4 datasets, build all ds columns, clean column names, prepare everything for forecasting and streamlit

In [90]:
# Ingestion and Normalisation Code for Multi-Channel Datasets
import pandas as pd

# =========================================================
# 1) DIGITAL CHANNELS (Website, Deliveroo, JustEat)
# =========================================================
digital_channels = pd.read_csv(
    r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Public_DigitalChannels_Monthly_Revenue_FinancialYear.csv"
)

# Extract first year from "2022-23"
digital_channels["Year"] = digital_channels["Financial_Year"].str.split("-").str[0]

# Build proper monthly timestamp
digital_channels["ds"] = pd.to_datetime(
    digital_channels["Year"] + "-" + digital_channels["Month"] + "-01",
    format="%Y-%B-%d"
)

# =========================================================
# 2) AGGREGATOR MONTHLY TOTALS
# =========================================================
aggregator_monthly = pd.read_csv(
    r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Public_Aggregator_Monthly_Revenue_FinancialYear.csv"
)

aggregator_monthly["Year"] = aggregator_monthly["Financial_Year"].str.split("-").str[0]

aggregator_monthly["ds"] = pd.to_datetime(
    aggregator_monthly["Year"] + "-" + aggregator_monthly["Month"] + "-01",
    format="%Y-%B-%d"
)

# =========================================================
# 3) DELIVEROO CATEGORY MONTHLY (FORECASTABLE)
# =========================================================
deliv_raw = pd.read_csv(
    r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Public_Deliveroo_Category_Summary_2023_2025.csv"
)

# Build proper monthly timestamp
deliv_raw["ds"] = pd.to_datetime(
    deliv_raw["Year"].astype(str) + "-" + deliv_raw["Month"] + "-01",
    format="%Y-%B-%d"
)

# Standardise revenue column name
deliv_raw = deliv_raw.rename(columns={"Gross_Revenue": "deliveroo_revenue"})

# =========================================================
# 4) EPOS ANNUAL CATEGORY MIX (INSIGHTS ONLY)
# =========================================================
epos_annual = pd.read_csv(
    r"C:\Users\omota\OneDrive - Solent University\COM726_DISSERTATION\Kaggle_Dataset_Download\Public_EPOS_Annual_Category_Mix_2023_2025.csv"
)

# No date column needed — this dataset is annual, not monthly


In [92]:
digital_channels.head()
aggregator_monthly.head()
deliv_raw.head()

,Year,Month,Category,Quantity,deliveroo_revenue,Sales_Channel,Data_Level,Data_Period,ds
0,2023,August,Appetisers,26,126.5,Deliveroo,Monthly_Category_Aggregate,Jan 2023 - Dec 2025,2023-08-01
1,2023,August,Drinks,3,7.0,Deliveroo,Monthly_Category_Aggregate,Jan 2023 - Dec 2025,2023-08-01
2,2023,August,Extras,51,163.0,Deliveroo,Monthly_Category_Aggregate,Jan 2023 - Dec 2025,2023-08-01
3,2023,August,Main Courses,41,301.5,Deliveroo,Monthly_Category_Aggregate,Jan 2023 - Dec 2025,2023-08-01
4,2023,August,Modifiers,53,57.5,Deliveroo,Monthly_Category_Aggregate,Jan 2023 - Dec 2025,2023-08-01


Forecasting Engine Script - Channel + Deliveroo Category Forecasts using Prophet

In [74]:
from prophet import Prophet
import pandas as pd

# =========================================================
# FORECASTING ENGINE — CHANNEL & CATEGORY FORECASTS
# =========================================================

def prepare_channel_data(df, channel_name):
    """
    Filters and prepares a single channel's data for Prophet.
    """
    channel_df = df[df["Channel"] == channel_name].copy()
    channel_df = channel_df[["ds", "Revenue"]].rename(columns={"Revenue": "y"})
    channel_df = channel_df.sort_values("ds")
    return channel_df


def forecast_channel(df, channel_name, periods=12):
    """
    Builds a Prophet model for a single sales channel.
    Returns the model and forecast dataframe.
    """
    data = prepare_channel_data(df, channel_name)

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive"
    )

    model.fit(data)

    future = model.make_future_dataframe(periods=periods, freq="MS")
    forecast = model.predict(future)

    forecast["Channel"] = channel_name
    return model, forecast


# =========================================================
# DELIVEROO CATEGORY FORECASTING
# =========================================================

def prepare_deliveroo_category(df, category_name):
    """
    Filters Deliveroo category data and prepares it for Prophet.
    """
    cat_df = df[df["Category"] == category_name].copy()
    cat_df = cat_df[["ds", "deliveroo_revenue"]].rename(columns={"deliveroo_revenue": "y"})
    cat_df = cat_df.sort_values("ds")
    return cat_df


def forecast_deliveroo_category(df, category_name, periods=12):
    """
    Builds a Prophet model for a single Deliveroo category.
    """
    data = prepare_deliveroo_category(df, category_name)

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive"
    )

    model.fit(data)

    future = model.make_future_dataframe(periods=periods, freq="MS")
    forecast = model.predict(future)

    forecast["Category"] = category_name
    return model, forecast


# =========================================================
# BULK FORECASTING HELPERS
# =========================================================

def forecast_all_channels(digital_channels, aggregator_monthly):
    """
    Runs Prophet for all channels:
    Website, Deliveroo, JustEat, Aggregator.
    """
    channels = digital_channels["Channel"].unique().tolist()
    results = []

    for ch in channels:
        _, fc = forecast_channel(digital_channels, ch)
        results.append(fc)

    # Aggregator totals
    agg_df = aggregator_monthly[["ds", "Revenue"]].rename(columns={"Revenue": "y"})
    agg_df = agg_df.sort_values("ds")

    agg_model = Prophet(yearly_seasonality=True)
    agg_model.fit(agg_df)

    future = agg_model.make_future_dataframe(periods=12, freq="MS")
    agg_fc = agg_model.predict(future)
    agg_fc["Channel"] = "Aggregator"

    results.append(agg_fc)

    return pd.concat(results, ignore_index=True)


def forecast_all_deliveroo_categories(deliv_raw):
    """
    Runs Prophet for every Deliveroo category.
    """
    categories = deliv_raw["Category"].unique().tolist()
    results = []

    for cat in categories:
        _, fc = forecast_deliveroo_category(deliv_raw, cat)
        results.append(fc)

    return pd.concat(results, ignore_index=True)


In [96]:
channel_forecasts = forecast_all_channels(digital_channels, aggregator_monthly)
category_forecasts = forecast_all_deliveroo_categories(deliv_raw)
channel_forecasts = forecast_all_channels(digital_channels, aggregator_monthly)
category_forecasts = forecast_all_deliveroo_categories(deliv_raw)
channel_forecasts.head()
category_forecasts.head()

20:14:39 - cmdstanpy - INFO - Chain [1] start processing
20:14:40 - cmdstanpy - INFO - Chain [1] done processing
20:14:41 - cmdstanpy - INFO - Chain [1] start processing
20:14:41 - cmdstanpy - INFO - Chain [1] done processing
20:14:41 - cmdstanpy - INFO - Chain [1] start processing
20:14:42 - cmdstanpy - INFO - Chain [1] done processing
20:14:42 - cmdstanpy - INFO - Chain [1] start processing
20:14:42 - cmdstanpy - INFO - Chain [1] done processing
20:14:43 - cmdstanpy - INFO - Chain [1] start processing
20:14:43 - cmdstanpy - INFO - Chain [1] done processing
20:14:43 - cmdstanpy - INFO - Chain [1] start processing
20:14:44 - cmdstanpy - INFO - Chain [1] done processing
20:14:44 - cmdstanpy - INFO - Chain [1] start processing
20:14:44 - cmdstanpy - INFO - Chain [1] done processing
20:14:45 - cmdstanpy - INFO - Chain [1] start processing
20:14:45 - cmdstanpy - INFO - Chain [1] done processing
20:14:46 - cmdstanpy - INFO - Chain [1] start processing
20:14:46 - cmdstanpy - INFO - Chain [1]

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat,Category
0,2023-05-01,113.520120,35.592659,121.641918,113.520120,113.520120,-33.915945,-33.915945,-33.915945,-33.915945,-33.915945,-33.915945,0.0,0.0,0.0,79.604175,Appetisers
1,2023-06-01,116.658655,76.285954,160.660407,116.658655,116.658655,3.289838,3.289838,3.289838,3.289838,3.289838,3.289838,0.0,0.0,0.0,119.948493,Appetisers
2,2023-07-01,119.695946,112.441348,196.423864,119.695946,119.695946,34.630796,34.630796,34.630796,34.630796,34.630796,34.630796,0.0,0.0,0.0,154.326743,Appetisers
3,2023-08-01,122.834481,71.929035,156.193616,122.834481,122.834481,-8.025715,-8.025715,-8.025715,-8.025715,-8.025715,-8.025715,0.0,0.0,0.0,114.808766,Appetisers
4,2023-09-01,125.973016,44.619880,131.936778,125.973016,125.973016,-38.015004,-38.015004,-38.015004,-38.015004,-38.015004,-38.015004,0.0,0.0,0.0,87.958012,Appetisers


In [100]:
channel_forecasts.head()

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,additive_terms,additive_terms_lower,additive_terms_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat,Channel
0,2022-01-01,909.416414,263.960924,603.961821,909.416414,909.416414,-477.472904,-477.472904,-477.472904,-477.472904,-477.472904,-477.472904,0.0,0.0,0.0,431.943510,Deliveroo
1,2022-02-01,906.387672,694.865900,1037.003627,906.387672,906.387672,-37.811004,-37.811004,-37.811004,-37.811004,-37.811004,-37.811004,0.0,0.0,0.0,868.576668,Deliveroo
2,2022-03-01,903.652034,726.772954,1069.785387,903.652034,903.652034,3.170619,3.170619,3.170619,3.170619,3.170619,3.170619,0.0,0.0,0.0,906.822653,Deliveroo
3,2023-01-01,873.755423,394.324017,755.137670,873.755423,873.755423,-290.800421,-290.800421,-290.800421,-290.800421,-290.800421,-290.800421,0.0,0.0,0.0,582.955002,Deliveroo
4,2023-02-01,870.726681,697.681120,1041.863002,870.726681,870.726681,-1.596198,-1.596198,-1.596198,-1.596198,-1.596198,-1.596198,0.0,0.0,0.0,869.130483,Deliveroo
